In [0]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("CONTAINER_NAME")

In [0]:
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"
path_rastreamento = base_path + "vendas_raw/2026/02/21/112200/ecommerce_rastreamento.parquet"

df_rastreamento = (
    spark.read
    .format("parquet")
    .option(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
    .option(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    .option(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
    .option(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
    .option(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
    .load(path_rastreamento)
)

display(df_rastreamento)

In [0]:
#Tipos de dados
df_rastreamento.printSchema()

In [0]:
display(
    spark.createDataFrame(
        df_rastreamento.dtypes,
        ["coluna", "tipo_dado"]
    )
)

In [0]:
#Contagem total de registros
total_registros = df_rastreamento.count()

print(f"Total de registros: {total_registros}")

In [0]:
#Contagem de nulos por coluna
from pyspark.sql.functions import col, sum as spark_sum, when

df_nulos = df_rastreamento.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_rastreamento.columns
])

display(df_nulos)

In [0]:
#Valores mínimos e máximos
from pyspark.sql.functions import min as spark_min, max as spark_max

df_min_max = df_rastreamento.select(
    spark_min("id_rastreamento").alias("id_rastreamento_min"),
    spark_max("id_rastreamento").alias("id_rastreamento_max"),

    spark_min("id_pedido_ecommerce").alias("id_pedido_ecommerce_min"),
    spark_max("id_pedido_ecommerce").alias("id_pedido_ecommerce_max"),

    spark_min("id_transportadora").alias("id_transportadora_min"),
    spark_max("id_transportadora").alias("id_transportadora_max"),

    spark_min("dt_evento").alias("dt_evento_min"),
    spark_max("dt_evento").alias("dt_evento_max")
)

display(df_min_max)

In [0]:
#Verificar chave primária
total = df_rastreamento.count()
distintos = df_rastreamento.select("id_rastreamento").distinct().count()
nulos_pk = df_rastreamento.filter(col("id_rastreamento").isNull()).count()

print(f"Total de registros: {total}")
print(f"IDs distintos: {distintos}")
print(f"IDs nulos: {nulos_pk}")

if total == distintos and nulos_pk == 0:
    print("id_rastreamento pode ser considerada chave primária.")
else:
    print("id_rastreamento NÃO pode ser considerada chave primária.")

# Análise Exploratória - Tabela ecommerce_rastreamento

## Objetivo

Realizar análise exploratória da tabela `ecommerce_rastreamento`, identificando sua estrutura, tipos de dados, qualidade das informações e possíveis chaves primárias.

---

## Descrição da Tabela

A tabela `ecommerce_rastreamento` armazena eventos relacionados ao acompanhamento da entrega de pedidos do e-commerce, permitindo monitorar o status logístico dos pedidos ao longo do tempo.

---

## Estrutura dos Dados

| Coluna | Tipo de Dado |
|----------|----------|
| id_rastreamento | long |
| id_pedido_ecommerce | long |
| codigo_rastreio | string |
| id_transportadora | long |
| status_entrega | string |
| dt_evento | timestamp_ntz |
| observacao | string |

---

## Análise de Valores Nulos

Foi realizada a contagem de valores nulos em todas as colunas da tabela para avaliar a qualidade dos dados e identificar possíveis inconsistências.

Os resultados podem ser visualizados na execução da célula de análise de nulos.

---

## Valores Mínimos e Máximos

Foram analisadas as colunas numéricas e temporais da tabela:

- id_rastreamento
- id_pedido_ecommerce
- id_transportadora
- dt_evento

Os valores mínimos e máximos encontrados estão apresentados na tabela gerada pelo notebook.

---

## Chave Primária

A coluna `id_rastreamento` foi avaliada como candidata à chave primária da tabela.

A validação foi realizada comparando a quantidade total de registros com a quantidade de valores distintos da coluna.

Caso o total de registros seja igual ao total de valores distintos e não existam valores nulos, a coluna pode ser considerada uma chave primária válida.

---

## Conclusão

A tabela `ecommerce_rastreamento` apresenta estrutura adequada para acompanhamento dos eventos de rastreamento dos pedidos do e-commerce.

Os dados possuem colunas de identificação, status logístico, informações de transportadora e registro temporal dos eventos, sendo adequados para análises operacionais e monitoramento em tempo real do processo de entrega.